# 06 — Simulator Measurements

Quantum measurement is fundamentally probabilistic.  In this notebook we explore:

- How to run a circuit many times (shots) to build up statistics
- How to read and interpret Qiskit measurement counts
- How shot noise converges to the theoretical probability as shots increase
- How to measure in different bases


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import plot_counts
setup_notebook()

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

sim = AerSimulator()

def run(qc, shots=1024):
    job = sim.run(qc.decompose(), shots=shots)
    return job.result().get_counts()


## Shot noise and convergence

As we increase the number of shots, the empirical frequencies converge to the true probabilities.

For $H|0\rangle = |+\rangle$, the true probability is $P(0) = P(1) = 0.5$.


In [ ]:
shot_counts_list = [10, 100, 1000, 10000]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for ax, shots in zip(axes, shot_counts_list):
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.measure_all()
    counts = run(qc, shots=shots)
    plot_counts(counts, title=f"{shots} shots", ax=ax)

plt.suptitle(r"$H|0angle$ — convergence of shot frequencies", y=1.02)
plt.tight_layout()
plt.show()


## Measuring in different bases

By default, Qiskit measures in the Z basis ($|0\rangle$/$|1\rangle$).  
To measure in the X basis ($|+\rangle$/$|-\rangle$) or Y basis, we apply a basis-change rotation before measurement.

| Basis | Rotation before measure |
|---|---|
| Z | none |
| X | H |
| Y | $S^\dagger H$ |


In [ ]:
def measure_in_basis(state_prep, basis="Z", shots=1024):
    """Prepare a state and measure in the given basis (Z, X, or Y)."""
    qc = QuantumCircuit(1)
    state_prep(qc)   # prepare the state
    if basis == "X":
        qc.h(0)
    elif basis == "Y":
        qc.sdg(0)
        qc.h(0)
    qc.measure_all()
    return run(qc, shots=shots)

# State: |+> (equator of Bloch sphere on X axis)
def prep_plus(qc):
    qc.h(0)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, basis in zip(axes, ["Z", "X", "Y"]):
    counts = measure_in_basis(prep_plus, basis=basis, shots=2048)
    plot_counts(counts, title=f"|+⟩ in {basis}-basis", ax=ax)

plt.suptitle("Measuring |+⟩ in three bases", y=1.02)
plt.tight_layout()
plt.show()


## Interpreting the results

- **Z basis**: $|+\rangle$ measured in Z gives 50/50 — because $|+\rangle$ has equal Z-basis amplitudes.
- **X basis**: $|+\rangle$ measured in X gives ~100% '0' — because $|+\rangle$ is an eigenstate of X.
- **Y basis**: $|+\rangle$ measured in Y gives 50/50 — because $|+\rangle$ has equal Y-basis amplitudes.

This is the **Bloch sphere geometry** expressed as measurement statistics.


In [ ]:
# Bell state as a two-qubit example
qc2 = QuantumCircuit(2)
qc2.h(0)
qc2.cx(0, 1)
qc2.measure_all()

counts2 = run(qc2, shots=2048)
print("Bell state |Φ+⟩ measurement counts:", counts2)

fig, ax = plt.subplots(figsize=(4, 3))
plot_counts(counts2, title=r"Bell state $|\Phi^+angle$ — only 00 and 11", ax=ax)
plt.tight_layout()
plt.show()


## Summary

- Shot noise decreases as $\sim 1/\sqrt{N_{\text{shots}}}$; use more shots for cleaner statistics.
- Different measurement bases reveal different aspects of the quantum state.
- The Bloch sphere geometry predicts exactly which bases give deterministic vs. random outcomes.

**Next:** [07_state_preparation_and_visual_checks.ipynb](07_state_preparation_and_visual_checks.ipynb)
